# ODI to Databricks Migration: `trg_loc_insert.sql`

**Conversion Timestamp:** 2024-07-30 12:00:00

**Description:** This notebook converts an ODI SQL script for inserting data into the `TRG_LOC` table from `LOCATIONS` table in the HR schema. It handles schema and table name conversions, removes Oracle-specific hints, and sets up standard Databricks ETL parameters and target table operations.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type (e.g., L for Load)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "0", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "0", "ETL Process WID")
dbutils.widgets.text("ODI_SESS_NO", "0", "ODI Session Number")

## ETL Parameters

In [ ]:
%sql
-- SCEN_TASK_NO {10}: (Empty task - preserving order)
-- SCEN_TASK_NO {20}: (Empty task - preserving order)
-- SCEN_TASK_NO {30}: (Empty task - preserving order)

-- Placeholder for ETL_LAST_EXTRACT_TIME. Assuming a default if not explicitly managed by a table.
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT to_timestamp('1900-01-01 00:00:00', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time;

In [ ]:
display(spark.sql("""
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  '${DATASOURCE_NUM_ID}' AS datasource_num_id,
  '${ETL_PROC_WID}' AS etl_proc_wid,
  '${ODI_SESS_NO}' AS odi_sess_no,
  v_etl_last_extract_time.etl_last_extract_time
FROM v_etl_last_extract_time
"""))

## Target Table: `workspace.hr.trg_loc`

In [ ]:
%sql
-- Create TRG_LOC table if it does not exist, inferred from LOCATIONS structure
CREATE TABLE IF NOT EXISTS workspace.hr.trg_loc (
    LOCATION_ID     BIGINT,
    STREET_ADDRESS  STRING,
    POSTAL_CODE     STRING,
    CITY            STRING,
    STATE_PROVINCE  STRING,
    COUNTRY_ID      STRING
) USING DELTA;

In [ ]:
%sql
-- Insert data into target table
-- Original SCEN_TASK_NO in {30} INSERT statement
-- Removed Oracle-specific hint /*+ APPEND PARALLEL */
INSERT INTO workspace.hr.trg_loc (
    LOCATION_ID,
    STREET_ADDRESS,
    POSTAL_CODE,
    CITY,
    STATE_PROVINCE,
    COUNTRY_ID
)
SELECT
    locations.LOCATION_ID,
    locations.STREET_ADDRESS,
    locations.POSTAL_CODE,
    locations.CITY,
    locations.STATE_PROVINCE,
    locations.COUNTRY_ID
FROM
    workspace.hr.locations AS locations;

In [ ]:
%sql
SELECT COUNT(*) AS record_count FROM workspace.hr.trg_loc;

## Optimize Target Table

In [ ]:
%sql
-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.hr.trg_loc ZORDER BY (LOCATION_ID, COUNTRY_ID);

## Validation

In [ ]:
%sql
SELECT 'TRG_LOC' AS table_name, COUNT(*) AS final_record_count FROM workspace.hr.trg_loc;

In [ ]:
%sql
SELECT * FROM workspace.hr.trg_loc TABLESAMPLE (10 ROWS) WHERE LOCATION_ID IS NOT NULL ORDER BY LOCATION_ID DESC;

## Conversion Notes & Manual Actions Required

1.  **Schema and Table Naming:** All Oracle schema references (`HR`) have been converted to `workspace.hr` and table names (`TRG_LOC`, `LOCATIONS`) to lowercase (`trg_loc`, `locations`).
2.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` hint has been removed as it's not applicable in Databricks Delta Lake.
3.  **Data Types:** Inferred data types for `trg_loc` table have been mapped from common Oracle HR schema types to Spark SQL types (e.g., `NUMBER` to `BIGINT`, `VARCHAR2` to `STRING`). Please verify these against your actual source DDL for `LOCATIONS`.
4.  **ETL Parameters:** Standard Databricks widgets have been introduced to capture common ETL parameters (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO`).
5.  **Temporary Views:** A `v_etl_last_extract_time` temporary view is included as a placeholder. In a real scenario, this would typically read from a persistent ETL control table.
6.  **Optimization:** An `OPTIMIZE ... ZORDER BY` statement has been added for `trg_loc` to improve query performance on common join/filter columns. The `spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled` flag is set to `false` to prevent errors if statistics are not fully collected.
7.  **Idempotency:** The `CREATE TABLE IF NOT EXISTS` ensures that the DDL for `trg_loc` runs only if the table doesn't already exist. The `INSERT` statement will append new records to `trg_loc` if it already contains data. If a full refresh/overwrite is intended, a `TRUNCATE TABLE` or `DELETE FROM` would be needed before the `INSERT`.
8.  **Empty SCEN_TASK_NOs:** The empty `SCEN_TASK_NO` blocks ({10}, {20}, {30}) have been preserved as comments within the nearest logical cell to maintain lineage, but they do not translate to executable code.